In [ ]:
import pandas as pd
import numpy as np
import gc
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import gc
sns.set(style="whitegrid")

In [ ]:
data = pd.read_csv("eeg_data.csv")

In [ ]:
eeg_data_columns = data.columns.values.tolist()

In [ ]:
print(data["label"].value_counts())
cleanup_rules = ({'NEUTRAL': 0, 'POSITIVE': 1, 'NEGATIVE': 2} )
eeg_data_encoded = data.replace(cleanup_rules, inplace=False)
print(eeg_data_encoded['label'].value_counts())

In [ ]:
import seaborn as sns
sns.countplot(x=eeg_data_encoded['label'], data=data)

In [ ]:
data.count()

In [ ]:
eeg_data_encoded['label'].unique()

In [ ]:
eeg_data_encoded.head()

In [ ]:
x=eeg_data_encoded.drop(["label"]  ,axis=1)
x.shape

In [ ]:
y = eeg_data_encoded.loc[:,'label'].values
y.shape

In [ ]:
import pandas as pd
print("DataFrame Information:")
print(eeg_data_encoded.info())
print("\nFirst few rows of the DataFrame:")
print(eeg_data_encoded.head())
print("\nDataFrame Description:")
print(eeg_data_encoded.describe())

# Pearson Correlation

In [ ]:
feature_name = x.columns.tolist()
def cor_selector(x, y):
    cor_list = []
    for i in x.columns.tolist():
        cor = np.corrcoef(x[i], y)[0, 1]
        cor_list.append(cor)
    cor_list = [0 if np.isnan(i) else i for i in cor_list]
    cor_feature = x.iloc[:,np.argsort(np.abs(cor_list))[-100:]].columns.tolist()
    cor_support = [True if i in cor_feature else False for i in feature_name]
    return cor_support, cor_feature

In [ ]:
cor_support, cor_feature = cor_selector(x, y)
print(str(len(cor_feature)), 'selected features')

In [ ]:
cor_feature

In [ ]:
from sklearn.model_selection import train_test_split
x_cor_feature=data[cor_feature]
x_cor_train, x_cor_test, y_cor_train, y_cor_test = train_test_split(x_cor_feature, y, test_size = 0.2)

# Chi-Squared

In [ ]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.preprocessing import MinMaxScaler
X_norm = MinMaxScaler().fit_transform(x)
chi_selector = SelectKBest(chi2, k=100)
chi_selector.fit(X_norm, y)

In [ ]:
chi_support = chi_selector.get_support()
chi_feature = x.loc[:,chi_support].columns.tolist()
print(str(len(chi_feature)), 'selected features')

In [ ]:
chi_feature

In [ ]:
from sklearn.model_selection import train_test_split
x_chi_feature=data[chi_feature]
x_chi_train, x_chi_test, y_chi_train, y_chi_test = train_test_split(x_chi_feature, y, test_size = 0.2)

# Recursive Feature Elimination

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
rfe_selector = RFE(estimator=LogisticRegression(), n_features_to_select=100, step=10, verbose=5)
rfe_selector.fit(X_norm, y)

In [ ]:
rfe_support = rfe_selector.get_support()
rfe_feature = x.loc[:,rfe_support].columns.tolist()
print(str(len(rfe_feature)), 'selected features')

In [ ]:
rfe_feature

In [ ]:
from sklearn.model_selection import train_test_split
x_rfe_feature=data[rfe_feature]
x_rfe_train, x_rfe_test, y_rfe_train, y_rfe_test = train_test_split(x_rfe_feature, y, test_size = 0.2)

# SelectFromModel - Logistic Regression

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression

embeded_lr_selector = SelectFromModel(LogisticRegression(penalty="l2"), threshold='1.25*median')

embeded_lr_selector.fit(X_norm, y)

In [ ]:
embeded_lr_support = embeded_lr_selector.get_support()
embeded_lr_feature = x.loc[:,embeded_lr_support].columns.tolist()
print(str(len(embeded_lr_feature)), 'selected features')

In [ ]:
embeded_lr_feature

In [ ]:
from sklearn.model_selection import train_test_split
x_embeded_lr_feature=data[embeded_lr_feature]
x_lr_train, x_lr_test, y_lr_train, y_lr_test = train_test_split(x_embeded_lr_feature, y, test_size = 0.2)

# SelectFromModel - RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

embeded_rf_selector = SelectFromModel(RandomForestClassifier(n_estimators=100), threshold='1.25*median')
embeded_rf_selector.fit(x, y)

In [ ]:
from sklearn.feature_selection import SelectFromModel
embeded_rf_support = embeded_rf_selector.get_support()
embeded_rf_feature = x.loc[:,embeded_rf_support].columns.tolist()
print(str(len(embeded_rf_feature)), 'selected features')

In [ ]:
embeded_rf_feature

In [ ]:
from sklearn.model_selection import train_test_split
x_embeded_rf_feature=data[embeded_rf_feature]
x_rf_train, x_rf_test, y_rf_train, y_rf_test = train_test_split(x_embeded_rf_feature, y, test_size = 0.2)

# SelectFromModel - LGBM

In [ ]:
from sklearn.feature_selection import SelectFromModel
from lightgbm import LGBMClassifier

lgbc=LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=32, colsample_bytree=0.2,
            reg_alpha=3, reg_lambda=1, min_split_gain=0.01, min_child_weight=40)

embeded_lgb_selector = SelectFromModel(lgbc, threshold='1.25*median')
embeded_lgb_selector.fit(x, y)

In [ ]:
embeded_lgb_support = embeded_lgb_selector.get_support()
embeded_lgb_feature = x.loc[:,embeded_lgb_support].columns.tolist()
print(str(len(embeded_lgb_feature)), 'selected features')

In [ ]:
embeded_lgb_feature

In [ ]:
from sklearn.model_selection import train_test_split
x_embeded_lgb_feature=data[embeded_lgb_feature]
x_lgb_train, x_lgb_test, y_lgb_train, y_lgb_test = train_test_split(x_embeded_lgb_feature, y, test_size = 0.2)

# KNN - Pearson Correlation

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn_corr = KNeighborsClassifier(n_neighbors=3)
knn_corr.fit(x_cor_train, y_cor_train)

In [ ]:
mean_accuracy=knn_corr.score(x_cor_test, y_cor_test)
y_pred = knn_corr.predict(x_cor_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_cor_test,y_pred))
print(classification_report(y_cor_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(knn_corr.score(x_cor_train, y_cor_train)))
print ("Testing Accuracy {}".format(knn_corr.score(x_cor_test, y_cor_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate confusion matrix
conf_matrix = confusion_matrix(y_cor_test, knn_corr.predict(x_cor_test))

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# Decision Tree - Pearson Correlation

In [ ]:
from sklearn.tree import DecisionTreeClassifier
decisiontree_corr = DecisionTreeClassifier(random_state=0)
decisiontree_corr.fit(x_cor_train, y_cor_train)

In [ ]:
mean_accuracy=decisiontree_corr.score(x_cor_test, y_cor_test)
y_pred = decisiontree_corr.predict(x_cor_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_cor_test,y_pred))
print(classification_report(y_cor_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(decisiontree_corr.score(x_cor_train, y_cor_train)))
print ("Testing Accuracy {}".format(decisiontree_corr.score(x_cor_test, y_cor_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate confusion matrix
conf_matrix = confusion_matrix(y_cor_test, knn_corr.predict(x_cor_test))

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# Random Forest - Pearson Correlation

In [ ]:
from sklearn.ensemble import RandomForestClassifier
randomforest_corr = RandomForestClassifier(n_estimators=100, max_depth=2, random_state=0)
randomforest_corr.fit(x_cor_train, y_cor_train)

In [ ]:
mean_accuracy=randomforest_corr.score(x_cor_test, y_cor_test)
y_pred = randomforest_corr.predict(x_cor_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_cor_test,y_pred))
print(classification_report(y_cor_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(randomforest_corr.score(x_cor_train, y_cor_train)))
print ("Testing Accuracy {}".format(randomforest_corr.score(x_cor_test, y_cor_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate confusion matrix
conf_matrix = confusion_matrix(y_cor_test, randomforest_corr.predict(x_cor_test))

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# Gaussian Naive Bayes - Pearson Correlation

In [ ]:
from sklearn.naive_bayes import GaussianNB
gaussian_corr = GaussianNB()
gaussian_corr.fit(x_cor_train, y_cor_train)

In [ ]:
mean_accuracy=gaussian_corr.score(x_cor_test, y_cor_test)
y_pred = gaussian_corr.predict(x_cor_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_cor_test,y_pred))
print(classification_report(y_cor_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(gaussian_corr.score(x_cor_train, y_cor_train)))
print ("Testing Accuracy {}".format(gaussian_corr.score(x_cor_test, y_cor_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate confusion matrix
conf_matrix = confusion_matrix(y_cor_test, gaussian_corr.predict(x_cor_test))

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# SVM - Pearson Correlation

In [ ]:
from sklearn import svm
support_corr = svm.LinearSVC(random_state=20)
support_corr.fit(x_cor_train, y_cor_train)

In [ ]:
mean_accuracy=support_corr.score(x_cor_test, y_cor_test)
y_pred = support_corr.predict(x_cor_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_cor_test,y_pred))
print(classification_report(y_cor_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(support_corr.score(x_cor_train, y_cor_train)))
print ("Testing Accuracy {}".format(support_corr.score(x_cor_test, y_cor_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = support_corr.predict(x_cor_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_cor_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
y=['KNN', 'Decision Tree', 'Random Forest', 'Gaussian Naive Bayes', 'SVM']
x = [knn_corr.score(x_cor_test, y_cor_test), decisiontree_corr.score(x_cor_test, y_cor_test), randomforest_corr.score(x_cor_test, y_cor_test), gaussian_corr.score(x_cor_test, y_cor_test), support_corr.score(x_cor_test, y_cor_test)]
plt.plot(x,y)

In [ ]:
import matplotlib.pyplot as plt
x = ['KNN', 'Decision Tree', 'Random Forest', 'Naive Bayes', 'SVM']
y = [knn_corr.score(x_cor_test, y_cor_test), decisiontree_corr.score(x_cor_test, y_cor_test), randomforest_corr.score(x_cor_test, y_cor_test), gaussian_corr.score(x_cor_test, y_cor_test), support_corr.score(x_cor_test, y_cor_test)]

x_pos = [i for i, _ in enumerate(x)]

fig, ax = plt.subplots()
rects1 = ax.bar(x_pos, y, color='#d9534f')
plt.xlabel("Algorithms")
plt.ylabel("Accuracy")
plt.title("Accuracy of ML algorithms using Correlation Feature Selection\n")
plt.xticks(x_pos, x,rotation =90)
# Turn on the grid
plt.minorticks_on()
plt.grid(which='major', linestyle='-', linewidth='0.5', color='red')
# Customize the minor grid
plt.grid(which='minor', linestyle=':', linewidth='0.3', color='black')
def autolabel(rects):
    """
    Attach a text label above each bar displaying its height
    """
    for rect in rects:
        height = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2., 1.05*height,
                '%f' % float(height),
        ha='center', va='bottom')
autolabel(rects1)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame({'' : ['KNN', 'Decision Tree', 'Random Forest', 'Naive Bayes', 'SVM'],
'Accuracy' : [knn_corr.score(x_cor_test, y_cor_test), decisiontree_corr.score(x_cor_test, y_cor_test), randomforest_corr.score(x_cor_test, y_cor_test), gaussian_corr.score(x_cor_test, y_cor_test), support_corr.score(x_cor_test, y_cor_test)]})
n=[knn_corr.score(x_cor_test, y_cor_test), decisiontree_corr.score(x_cor_test, y_cor_test), randomforest_corr.score(x_cor_test, y_cor_test), gaussian_corr.score(x_cor_test, y_cor_test), support_corr.score(x_cor_test, y_cor_test)]
df = df.sort_values('Accuracy')

fig, ax = plt.subplots()
df.plot(kind='bar', x='', y='Accuracy', ax=ax, color='orange')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame({'' : ['KNN', 'Decision Tree', 'Random Forest', 'Naive Bayes', 'SVM'],
'Accuracy' : [knn_corr.score(x_cor_test, y_cor_test), decisiontree_corr.score(x_cor_test, y_cor_test), randomforest_corr.score(x_cor_test, y_cor_test), gaussian_corr.score(x_cor_test, y_cor_test), support_corr.score(x_cor_test, y_cor_test)]})
df = df.sort_values('Accuracy')

fig, ax = plt.subplots()
df.plot(kind='scatter', x='', y='Accuracy', ax=ax, color='red')
plt.show()

df.plot(kind='line', x='', y='Accuracy', ax=ax, color='orange')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame({'' : ['KNN', 'Decision Tree', 'Random Forest', 'Naive Bayes', 'SVM'],
'Accuracy' : [knn_corr.score(x_cor_test, y_cor_test), decisiontree_corr.score(x_cor_test, y_cor_test), randomforest_corr.score(x_cor_test, y_cor_test), gaussian_corr.score(x_cor_test, y_cor_test), support_corr.score(x_cor_test, y_cor_test)]})
n=[knn_corr.score(x_cor_test, y_cor_test), decisiontree_corr.score(x_cor_test, y_cor_test), randomforest_corr.score(x_cor_test, y_cor_test), gaussian_corr.score(x_cor_test, y_cor_test), support_corr.score(x_cor_test, y_cor_test)]
df = df.sort_values('Accuracy')

fig, ax = plt.subplots()
df.plot(kind='line', x='', y='Accuracy', ax=ax, color='orange')
plt.show()

In [ ]:
plt.scatter(x,y)

# KNN - Chi2

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn_chi = KNeighborsClassifier(n_neighbors=3)
knn_chi.fit(x_chi_train, y_chi_train)

In [ ]:
mean_accuracy=knn_chi.score(x_chi_test, y_chi_test)
y_pred = knn_chi.predict(x_chi_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_chi_test,y_pred))
print(classification_report(y_chi_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(knn_chi.score(x_chi_train, y_chi_train)))
print ("Testing Accuracy {}".format(knn_chi.score(x_chi_test, y_chi_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = knn_chi.predict(x_chi_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_chi_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# Decision Tree - Chi2

In [ ]:
from sklearn.tree import DecisionTreeClassifier
decisiontree_chi = DecisionTreeClassifier(random_state=0)
decisiontree_chi.fit(x_chi_train, y_chi_train)

In [ ]:
mean_accuracy=decisiontree_chi.score(x_chi_test, y_chi_test)
y_pred = decisiontree_chi.predict(x_chi_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_chi_test,y_pred))
print(classification_report(y_chi_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(decisiontree_chi.score(x_chi_train, y_chi_train)))
print ("Testing Accuracy {}".format(decisiontree_chi.score(x_chi_test, y_chi_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = decisiontree_chi.predict(x_chi_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_chi_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# Random Forest - Chi2

In [ ]:
from sklearn.ensemble import RandomForestClassifier
randomforest_chi = RandomForestClassifier(n_estimators=100, max_depth=2, random_state=0)
randomforest_chi.fit(x_chi_train, y_chi_train)

In [ ]:
mean_accuracy=randomforest_chi.score(x_chi_test, y_chi_test)
y_pred = randomforest_chi.predict(x_chi_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_chi_test,y_pred))
print(classification_report(y_chi_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(randomforest_chi.score(x_chi_train, y_chi_train)))
print ("Testing Accuracy {}".format(randomforest_chi.score(x_chi_test, y_chi_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = randomforest_chi.predict(x_chi_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_chi_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# Gaussian Naive Bayes - Chi2

In [ ]:
from sklearn.naive_bayes import GaussianNB
gaussian_chi = GaussianNB()
gaussian_chi.fit(x_chi_train, y_chi_train)

In [ ]:
mean_accuracy=gaussian_chi.score(x_chi_test, y_chi_test)
y_pred = gaussian_chi.predict(x_chi_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_chi_test,y_pred))
print(classification_report(y_chi_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(gaussian_chi.score(x_chi_train, y_chi_train)))
print ("Testing Accuracy {}".format(gaussian_chi.score(x_chi_test, y_chi_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = gaussian_chi.predict(x_chi_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_chi_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# SVM - Chi2

In [ ]:
from sklearn import svm
support_chi = svm.LinearSVC(random_state=20)
support_chi.fit(x_chi_train, y_chi_train)

In [ ]:
mean_accuracy=support_chi.score(x_chi_test, y_chi_test)
y_pred = support_chi.predict(x_chi_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_chi_test,y_pred))
print(classification_report(y_chi_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(support_chi.score(x_chi_train, y_chi_train)))
print ("Testing Accuracy {}".format(support_chi.score(x_chi_test, y_chi_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = support_chi.predict(x_chi_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_chi_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
y=['KNN', 'Decision Tree', 'Random Forest', 'Gaussian Naive Bayes', 'SVM']
x = [knn_chi.score(x_chi_test, y_chi_test), decisiontree_chi.score(x_chi_test, y_chi_test), randomforest_chi.score(x_chi_test, y_chi_test), gaussian_chi.score(x_chi_test, y_chi_test), support_chi.score(x_chi_test, y_chi_test)]
plt.plot(x,y)

In [ ]:
plt.scatter(x,y)

In [ ]:
import matplotlib.pyplot as plt
x=['KNN', 'Decision Tree', 'Random Forest', 'Naive Bayes', 'SVM']
y = [knn_chi.score(x_chi_test, y_chi_test), decisiontree_chi.score(x_chi_test, y_chi_test), randomforest_chi.score(x_chi_test, y_chi_test), gaussian_chi.score(x_chi_test, y_chi_test), support_chi.score(x_chi_test, y_chi_test)]

x_pos = [i for i, _ in enumerate(x)]

fig, ax = plt.subplots()
rects1 = ax.bar(x_pos, y, color='#d9534f')
plt.xlabel("Algorithms")
plt.ylabel("Accuracy")
plt.title("Accuracy of ML algorithms using Chi-Squared Feature Selection\n")
plt.xticks(x_pos, x,rotation =90)
# Turn on the grid
plt.minorticks_on()
plt.grid(which='major', linestyle='-', linewidth='0.5', color='red')
# Customize the minor grid
plt.grid(which='minor', linestyle=':', linewidth='0.3', color='black')
def autolabel(rects):
    """
    Attach a text label above each bar displaying its height
    """
    for rect in rects:
        height = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2., 1.05*height,
                '%f' % float(height),
        ha='center', va='bottom')
autolabel(rects1)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame({'' : ['KNN', 'Decision Tree', 'Random Forest', 'Naive Bayes', 'SVM'],
'Accuracy' : [knn_chi.score(x_chi_test, y_chi_test), decisiontree_chi.score(x_chi_test, y_chi_test), randomforest_chi.score(x_chi_test, y_chi_test), gaussian_chi.score(x_chi_test, y_chi_test), support_chi.score(x_chi_test, y_chi_test)]
})
df = df.sort_values('Accuracy')

fig, ax = plt.subplots()
df.plot(kind='scatter', x='', y='Accuracy', ax=ax, color='red')
plt.show()

df.plot(kind='line', x='', y='Accuracy', ax=ax, color='orange')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame({'' : ['KNN', 'Decision Tree', 'Random Forest', 'Naive Bayes', 'SVM'],
'Accuracy' : [knn_chi.score(x_chi_test, y_chi_test), decisiontree_chi.score(x_chi_test, y_chi_test), randomforest_chi.score(x_chi_test, y_chi_test), gaussian_chi.score(x_chi_test, y_chi_test), support_chi.score(x_chi_test, y_chi_test)]
})
df = df.sort_values('Accuracy')

fig, ax = plt.subplots()
df.plot(kind='line', x='', y='Accuracy', ax=ax, color='red')
plt.show()

df.plot(kind='line', x='', y='Accuracy', ax=ax, color='orange')
plt.show()

# KNN - RFE

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn_rfe = KNeighborsClassifier(n_neighbors=3)
knn_rfe.fit(x_rfe_train, y_rfe_train)

In [ ]:
mean_accuracy=knn_rfe.score(x_rfe_test, y_rfe_test)
y_pred = knn_rfe.predict(x_rfe_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_rfe_test,y_pred))
print(classification_report(y_rfe_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(knn_rfe.score(x_rfe_train, y_rfe_train)))
print ("Testing Accuracy {}".format(knn_rfe.score(x_rfe_test, y_rfe_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = knn_rfe.predict(x_rfe_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_rfe_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# DecisionTree - RFE

In [ ]:
from sklearn.tree import DecisionTreeClassifier
decisiontree_rfe = DecisionTreeClassifier(random_state=0)
decisiontree_rfe.fit(x_rfe_train, y_rfe_train)

In [ ]:
mean_accuracy=decisiontree_rfe.score(x_rfe_test, y_rfe_test)
y_pred = decisiontree_rfe.predict(x_rfe_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_rfe_test,y_pred))
print(classification_report(y_rfe_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(decisiontree_rfe.score(x_rfe_train, y_rfe_train)))
print ("Testing Accuracy {}".format(decisiontree_rfe.score(x_rfe_test, y_rfe_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = decisiontree_rfe.predict(x_rfe_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_rfe_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# RandomForest - RFE

In [ ]:
from sklearn.ensemble import RandomForestClassifier
randomforest_rfe = RandomForestClassifier(n_estimators=100, max_depth=2, random_state=0)
randomforest_rfe.fit(x_rfe_train, y_rfe_train)

In [ ]:
mean_accuracy=randomforest_rfe.score(x_rfe_test, y_rfe_test)
y_pred = randomforest_rfe.predict(x_rfe_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_rfe_test,y_pred))
print(classification_report(y_rfe_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(randomforest_rfe.score(x_rfe_train, y_rfe_train)))
print ("Testing Accuracy {}".format(randomforest_rfe.score(x_rfe_test, y_rfe_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = randomforest_rfe.predict(x_rfe_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_rfe_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# Gaussian Naive Bayes - RFE

In [ ]:
from sklearn.naive_bayes import GaussianNB
gaussian_rfe = GaussianNB()
gaussian_rfe.fit(x_rfe_train, y_rfe_train)

In [ ]:
mean_accuracy=gaussian_rfe.score(x_rfe_test, y_rfe_test)
y_pred = gaussian_rfe.predict(x_rfe_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_rfe_test,y_pred))
print(classification_report(y_rfe_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(gaussian_rfe.score(x_rfe_train, y_rfe_train)))
print ("Testing Accuracy {}".format(gaussian_rfe.score(x_rfe_test, y_rfe_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = gaussian_rfe.predict(x_rfe_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_rfe_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# SVM - RFE

In [ ]:
from sklearn import svm
support_rfe = svm.LinearSVC(random_state=20)
support_rfe.fit(x_rfe_train, y_rfe_train)

In [ ]:
mean_accuracy=support_rfe.score(x_rfe_test, y_rfe_test)
y_pred = support_rfe.predict(x_rfe_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_rfe_test,y_pred))
print(classification_report(y_rfe_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(support_rfe.score(x_rfe_train, y_rfe_train)))
print ("Testing Accuracy {}".format(support_rfe.score(x_rfe_test, y_rfe_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = support_rfe.predict(x_rfe_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_rfe_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
y=['KNN', 'Decision Tree', 'Random Forest', 'Gaussian Naive Bayes', 'SVM']
x = [knn_rfe.score(x_rfe_test, y_rfe_test), decisiontree_rfe.score(x_rfe_test, y_rfe_test), randomforest_rfe.score(x_rfe_test, y_rfe_test), gaussian_rfe.score(x_rfe_test, y_rfe_test), support_rfe.score(x_rfe_test, y_rfe_test)]
plt.plot(x,y)

In [ ]:
plt.scatter(x,y)

In [ ]:
import matplotlib.pyplot as plt
x=['KNN', 'Decision Tree', 'Random Forest', 'Naive Bayes', 'SVM']
y = [knn_rfe.score(x_rfe_test, y_rfe_test), decisiontree_rfe.score(x_rfe_test, y_rfe_test), randomforest_rfe.score(x_rfe_test, y_rfe_test), gaussian_rfe.score(x_rfe_test, y_rfe_test), support_rfe.score(x_rfe_test, y_rfe_test)]

x_pos = [i for i, _ in enumerate(x)]

fig, ax = plt.subplots()
rects1 = ax.bar(x_pos, y, color='#d9534f')
plt.xlabel("Algorithms")
plt.ylabel("Accuracy")
plt.title("Accuracy of ML algorithms using RFE Feature Selection\n")
plt.xticks(x_pos, x,rotation =90)
# Turn on the grid
plt.minorticks_on()
plt.grid(which='major', linestyle='-', linewidth='0.5', color='red')
# Customize the minor grid
plt.grid(which='minor', linestyle=':', linewidth='0.3', color='black')
def autolabel(rects):
    """
    Attach a text label above each bar displaying its height
    """
    for rect in rects:
        height = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2., 1.05*height,
                '%f' % float(height),
        ha='center', va='bottom')
autolabel(rects1)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame({'' : ['KNN', 'Decision Tree', 'Random Forest', 'Naive Bayes', 'SVM'],
'Accuracy' : [knn_rfe.score(x_rfe_test, y_rfe_test), decisiontree_rfe.score(x_rfe_test, y_rfe_test), randomforest_rfe.score(x_rfe_test, y_rfe_test), gaussian_rfe.score(x_rfe_test, y_rfe_test), support_rfe.score(x_rfe_test, y_rfe_test)]})
df = df.sort_values('Accuracy')

fig, ax = plt.subplots()
df.plot(kind='scatter', x='', y='Accuracy', ax=ax, color='red')
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame({'' : ['KNN', 'Decision Tree', 'Random Forest', 'Naive Bayes', 'SVM'],
'Accuracy' : [knn_rfe.score(x_rfe_test, y_rfe_test), decisiontree_rfe.score(x_rfe_test, y_rfe_test), randomforest_rfe.score(x_rfe_test, y_rfe_test), gaussian_rfe.score(x_rfe_test, y_rfe_test), support_rfe.score(x_rfe_test, y_rfe_test)]})
df = df.sort_values('Accuracy')

fig, ax = plt.subplots()


df.plot(kind='line', x='', y='Accuracy', ax=ax, color='orange')
plt.show()

# KNN - LR 

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn_lr = KNeighborsClassifier(n_neighbors=3)
knn_lr.fit(x_lr_train, y_lr_train)

In [ ]:
mean_accuracy=knn_lr.score(x_lr_test, y_lr_test)
y_pred = knn_lr.predict(x_lr_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_lr_test,y_pred))
print(classification_report(y_lr_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(knn_lr.score(x_lr_train, y_lr_train)))
print ("Testing Accuracy {}".format(knn_lr.score(x_lr_test, y_lr_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = knn_lr.predict(x_lr_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_lr_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# Decision Tree - LR

In [ ]:
from sklearn.tree import DecisionTreeClassifier
decisiontree_lr = DecisionTreeClassifier(random_state=0)
decisiontree_lr.fit(x_lr_train, y_lr_train)

In [ ]:
mean_accuracy=decisiontree_lr.score(x_lr_test, y_lr_test)
y_pred = decisiontree_lr.predict(x_lr_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_lr_test,y_pred))
print(classification_report(y_lr_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(decisiontree_lr.score(x_lr_train, y_lr_train)))
print ("Testing Accuracy {}".format(decisiontree_lr.score(x_lr_test, y_lr_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = decisiontree_lr.predict(x_lr_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_lr_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# Random Forest - LR

In [ ]:
from sklearn.ensemble import RandomForestClassifier
randomforest_lr = RandomForestClassifier(n_estimators=100, max_depth=2, random_state=0)
randomforest_lr.fit(x_lr_train, y_lr_train)

In [ ]:
mean_accuracy=randomforest_lr.score(x_lr_test, y_lr_test)
y_pred = randomforest_lr.predict(x_lr_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_lr_test,y_pred))
print(classification_report(y_lr_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(randomforest_lr.score(x_lr_train, y_lr_train)))
print ("Testing Accuracy {}".format(randomforest_lr.score(x_lr_test, y_lr_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = randomforest_lr.predict(x_lr_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_lr_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# Gaussian Naive Bayes - LR

In [ ]:
from sklearn.naive_bayes import GaussianNB
gaussian_lr = GaussianNB()
gaussian_lr.fit(x_lr_train, y_lr_train)

In [ ]:
mean_accuracy=gaussian_lr.score(x_lr_test, y_lr_test)
y_pred = gaussian_lr.predict(x_lr_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_lr_test,y_pred))
print(classification_report(y_lr_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(gaussian_lr.score(x_lr_train, y_lr_train)))
print ("Testing Accuracy {}".format(gaussian_lr.score(x_lr_test, y_lr_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = gaussian_lr.predict(x_lr_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_lr_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# SVM - LR

In [ ]:
from sklearn import svm
support_lr = svm.LinearSVC(random_state=20)
support_lr.fit(x_lr_train, y_lr_train)

In [ ]:
mean_accuracy=support_lr.score(x_lr_test, y_lr_test)
y_pred = support_lr.predict(x_lr_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_lr_test,y_pred))
print(classification_report(y_lr_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(support_lr.score(x_lr_train, y_lr_train)))
print ("Testing Accuracy {}".format(support_lr.score(x_lr_test, y_lr_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = support_lr.predict(x_lr_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_lr_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
y=['KNN', 'Decision Tree', 'Random Forest', 'Gaussian Naive Bayes', 'SVM']
x = [knn_lr.score(x_lr_test, y_lr_test), decisiontree_lr.score(x_lr_test, y_lr_test), randomforest_lr.score(x_lr_test, y_lr_test), gaussian_lr.score(x_lr_test, y_lr_test), support_lr.score(x_lr_test, y_lr_test)]
plt.plot(x,y)

In [ ]:
plt.scatter(x,y)

In [ ]:
import matplotlib.pyplot as plt
x=['KNN', 'Decision Tree', 'Random Forest', 'Naive Bayes', 'SVM']
y = [knn_lr.score(x_lr_test, y_lr_test), decisiontree_lr.score(x_lr_test, y_lr_test), randomforest_lr.score(x_lr_test, y_lr_test), gaussian_lr.score(x_lr_test, y_lr_test), support_lr.score(x_lr_test, y_lr_test)]

x_pos = [i for i, _ in enumerate(x)]

fig, ax = plt.subplots()
rects1 = ax.bar(x_pos, y, color='#d9534f')
plt.xlabel("Algorithms")
plt.ylabel("Accuracy")
plt.title("Accuracy of ML algorithms using LR Feature Selection\n")
plt.xticks(x_pos, x,rotation =90)
# Turn on the grid
plt.minorticks_on()
plt.grid(which='major', linestyle='-', linewidth='0.5', color='red')
# Customize the minor grid
plt.grid(which='minor', linestyle=':', linewidth='0.3', color='black')
def autolabel(rects):
    """
    Attach a text label above each bar displaying its height
    """
    for rect in rects:
        height = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2., 1.05*height,
                '%f' % float(height),
        ha='center', va='bottom')
autolabel(rects1)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame({'' : ['KNN', 'Decision Tree', 'Random Forest', 'Naive Bayes', 'SVM'],
'Accuracy' : [knn_lr.score(x_lr_test, y_lr_test), decisiontree_lr.score(x_lr_test, y_lr_test), randomforest_lr.score(x_lr_test, y_lr_test), gaussian_lr.score(x_lr_test, y_lr_test), support_lr.score(x_lr_test, y_lr_test)]})
df = df.sort_values('Accuracy')

fig, ax = plt.subplots()
df.plot(kind='scatter', x='', y='Accuracy', ax=ax, color='red')
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame({'' : ['KNN', 'Decision Tree', 'Random Forest', 'Naive Bayes', 'SVM'],
'Accuracy' : [knn_lr.score(x_lr_test, y_lr_test), decisiontree_lr.score(x_lr_test, y_lr_test), randomforest_lr.score(x_lr_test, y_lr_test), gaussian_lr.score(x_lr_test, y_lr_test), support_lr.score(x_lr_test, y_lr_test)]})
df = df.sort_values('Accuracy')

fig, ax = plt.subplots()


df.plot(kind='line', x='', y='Accuracy', ax=ax, color='orange')
plt.show()

# KNN - RF

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn_rf = KNeighborsClassifier(n_neighbors=3)
knn_rf.fit(x_rf_train, y_rf_train)

In [ ]:
mean_accuracy=knn_rf.score(x_rf_test, y_rf_test)
y_pred = knn_rf.predict(x_rf_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_rf_test,y_pred))
print(classification_report(y_rf_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(knn_rf.score(x_rf_train, y_rf_train)))
print ("Testing Accuracy {}".format(knn_rf.score(x_rf_test, y_rf_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = knn_rf.predict(x_rf_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_rf_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# Decision Tree - RF

In [ ]:
from sklearn.tree import DecisionTreeClassifier
decisiontree_rf = DecisionTreeClassifier(random_state=0)
decisiontree_rf.fit(x_rf_train, y_rf_train)

In [ ]:
mean_accuracy=decisiontree_rf.score(x_rf_test, y_rf_test)
y_pred = decisiontree_rf.predict(x_rf_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_rf_test,y_pred))
print(classification_report(y_rf_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(decisiontree_rf.score(x_rf_train, y_rf_train)))
print ("Testing Accuracy {}".format(decisiontree_rf.score(x_rf_test, y_rf_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = decisiontree_rf.predict(x_rf_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_rf_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# Random Forest - RF

In [ ]:
from sklearn.ensemble import RandomForestClassifier
randomforest_rf = RandomForestClassifier(n_estimators=100, max_depth=2, random_state=0)
randomforest_rf.fit(x_rf_train, y_rf_train)

In [ ]:
mean_accuracy=randomforest_rf.score(x_rf_test, y_rf_test)
y_pred = randomforest_rf.predict(x_rf_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_rf_test,y_pred))
print(classification_report(y_rf_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(randomforest_rf.score(x_rf_train, y_rf_train)))
print ("Testing Accuracy {}".format(randomforest_rf.score(x_rf_test, y_rf_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = randomforest_rf.predict(x_rf_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_rf_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# Gaussian Naive Bayes - RF

In [ ]:
from sklearn.naive_bayes import GaussianNB
gaussian_rf = GaussianNB()
gaussian_rf.fit(x_rf_train, y_rf_train)

In [ ]:
mean_accuracy=gaussian_rf.score(x_rf_test, y_rf_test)
y_pred = gaussian_rf.predict(x_rf_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_rf_test,y_pred))
print(classification_report(y_rf_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(gaussian_rf.score(x_rf_train, y_rf_train)))
print ("Testing Accuracy {}".format(gaussian_rf.score(x_rf_test, y_rf_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = gaussian_rf.predict(x_rf_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_rf_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# SVM - RF

In [ ]:
from sklearn import svm
support_rf = svm.LinearSVC(random_state=20)
support_rf.fit(x_rf_train, y_rf_train)

In [ ]:
mean_accuracy=support_rf.score(x_rf_test, y_rf_test)
y_pred = support_rf.predict(x_rf_test)
print(mean_accuracy)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_rf_test,y_pred))
print(classification_report(y_rf_test,y_pred))

In [ ]:
print ("Training Accuracy {}".format(support_rf.score(x_rf_train, y_rf_train)))
print ("Testing Accuracy {}".format(support_rf.score(x_rf_test, y_rf_test)))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions
predictions = support_rf.predict(x_rf_test)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_rf_test, predictions)

# Plot confusion matrix
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
y=['KNN', 'Decision Tree', 'Random Forest', 'Gaussian Naive Bayes', 'SVM']
x = [knn_rf.score(x_rf_test, y_rf_test), decisiontree_rf.score(x_rf_test, y_rf_test), randomforest_rf.score(x_rf_test, y_rf_test), gaussian_rf.score(x_rf_test, y_rf_test), support_rf.score(x_rf_test, y_rf_test)]
plt.plot(x,y)

In [ ]:
plt.scatter(x,y)

In [ ]:
import matplotlib.pyplot as plt
x=['KNN', 'Decision Tree', 'Random Forest', 'Naive Bayes', 'SVM']
y = [knn_rf.score(x_rf_test, y_rf_test), decisiontree_rf.score(x_rf_test, y_rf_test), randomforest_rf.score(x_rf_test, y_rf_test), gaussian_rf.score(x_rf_test, y_rf_test), support_rf.score(x_rf_test, y_rf_test)]

x_pos = [i for i, _ in enumerate(x)]

fig, ax = plt.subplots()
rects1 = ax.bar(x_pos, y, color='#d9534f')
plt.xlabel("Algorithms")
plt.ylabel("Accuracy")
plt.title("Accuracy of ML algorithms using RF Feature Selection\n")
plt.xticks(x_pos, x,rotation =90)
# Turn on the grid
plt.minorticks_on()
plt.grid(which='major', linestyle='-', linewidth='0.5', color='red')
# Customize the minor grid
plt.grid(which='minor', linestyle=':', linewidth='0.3', color='black')
def autolabel(rects):
    """
    Attach a text label above each bar displaying its height
    """
    for rect in rects:
        height = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2., 1.05*height,
                '%f' % float(height),
        ha='center', va='bottom')
autolabel(rects1)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame({'' : ['KNN', 'Decision Tree', 'Random Forest', 'Naive Bayes', 'SVM'],
'Accuracy' : [knn_rf.score(x_rf_test, y_rf_test), decisiontree_rf.score(x_rf_test, y_rf_test), randomforest_rf.score(x_rf_test, y_rf_test), gaussian_rf.score(x_rf_test, y_rf_test), support_rf.score(x_rf_test, y_rf_test)]
})
df = df.sort_values('Accuracy')

fig, ax = plt.subplots()


df.plot(kind='line', x='', y='Accuracy', ax=ax, color='orange')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame({'' : ['KNN', 'Decision Tree', 'Random Forest', 'Naive Bayes', 'SVM'],
'Accuracy' : [knn_rf.score(x_rf_test, y_rf_test), decisiontree_rf.score(x_rf_test, y_rf_test), randomforest_rf.score(x_rf_test, y_rf_test), gaussian_rf.score(x_rf_test, y_rf_test), support_rf.score(x_rf_test, y_rf_test)]
})
df = df.sort_values('Accuracy')

fig, ax = plt.subplots()


df.plot(kind='scatter', x='', y='Accuracy', ax=ax, color='orange')
plt.show()

In [ ]:
print ("KNN_COR : {}".format(knn_corr.score(x_cor_test, y_cor_test)))
print ("KNN_CHI : {}".format(knn_chi.score(x_chi_test, y_chi_test)))
print ("KNN_RFE : {}".format(knn_rfe.score(x_rfe_test, y_rfe_test)))
print ("KNN_LR  : {}".format(knn_lr.score(x_lr_test, y_lr_test)))
print ("KNN_RF  : {}".format(knn_rf.score(x_rf_test, y_rf_test)))

print ("DecisionTree_COR : {}".format(decisiontree_corr.score(x_cor_test, y_cor_test)))
print ("DecisionTree_CHI : {}".format(decisiontree_chi.score(x_chi_test, y_chi_test)))
print ("DecisionTree_RFE : {}".format(decisiontree_rfe.score(x_rfe_test, y_rfe_test)))
print ("DecisionTree_LR  : {}".format(decisiontree_lr.score(x_lr_test, y_lr_test)))
print ("DecisionTree_RF  : {}".format(decisiontree_rf.score(x_rf_test, y_rf_test)))

print ("RandomForest_COR : {}".format(randomforest_corr.score(x_cor_test, y_cor_test)))
print ("RandomForest_CHI : {}".format(randomforest_chi.score(x_chi_test, y_chi_test)))
print ("RandomForest_RFE : {}".format(randomforest_rfe.score(x_rfe_test, y_rfe_test)))
print ("RandomForest_LR  : {}".format(randomforest_lr.score(x_lr_test, y_lr_test)))
print ("RandomForest_RF  : {}".format(randomforest_rf.score(x_rf_test, y_rf_test)))

print ("GaussianNaiveBayes_COR : {}".format(gaussian_corr.score(x_cor_test, y_cor_test)))
print ("GaussianNaiveBayes_CHI : {}".format(gaussian_chi.score(x_chi_test, y_chi_test)))
print ("GaussianNaiveBayes_RFE : {}".format(gaussian_rfe.score(x_rfe_test, y_rfe_test)))
print ("GaussianNaiveBayes_LR  : {}".format(gaussian_lr.score(x_lr_test, y_lr_test)))
print ("GaussianNaiveBayes_RF  : {}".format(gaussian_rf.score(x_rf_test, y_rf_test)))

print ("SVM_COR : {}".format(support_corr.score(x_cor_test, y_cor_test)))
print ("SVM_CHI : {}".format(support_chi.score(x_chi_test, y_chi_test)))
print ("SVM_RFE : {}".format(support_rfe.score(x_rfe_test, y_rfe_test)))
print ("SVM_LR  : {}".format(support_lr.score(x_lr_test, y_lr_test)))
print ("SVM_RF  : {}".format(support_rf.score(x_rf_test, y_rf_test)))

In [ ]:
y=['KNN_COR', 'DecisionTree_COR', 'RandomForest_COR', 'GaussianNaiveBayes_COR', 'SVM_COR',
  'KNN_CHI', 'DecisionTree_CHI', 'RandomForest_CHI', 'GaussianNaiveBayes_CHI', 'SVM_CHI',
   'KNN_RFE', 'DecisionTree_RFE', 'RandomForest_RFE', 'GaussianNaiveBayes_RFE', 'SVM_RFE',
   'KNN_LR', 'DecisionTree_LR', 'RandomForest_LR', 'GaussianNaiveBayes_LR', 'SVM_LR',
   'KNN_RF', 'DecisionTree_RF', 'RandomForest_RF', 'GaussianNaiveBayes_RF', 'SVM_RF']
x = [knn_corr.score(x_cor_test, y_cor_test), decisiontree_corr.score(x_cor_test, y_cor_test), randomforest_corr.score(x_cor_test, y_cor_test), gaussian_corr.score(x_cor_test, y_cor_test), support_corr.score(x_cor_test, y_cor_test),
     knn_chi.score(x_chi_test, y_chi_test), decisiontree_chi.score(x_chi_test, y_chi_test), randomforest_chi.score(x_chi_test, y_chi_test), gaussian_chi.score(x_chi_test, y_chi_test), support_chi.score(x_chi_test, y_chi_test),
     knn_rfe.score(x_rfe_test, y_rfe_test), decisiontree_rfe.score(x_rfe_test, y_rfe_test), randomforest_rfe.score(x_rfe_test, y_rfe_test), gaussian_rfe.score(x_rfe_test, y_rfe_test), support_rfe.score(x_rfe_test, y_rfe_test),
knn_lr.score(x_lr_test, y_lr_test), decisiontree_lr.score(x_lr_test, y_lr_test), randomforest_lr.score(x_lr_test, y_lr_test), gaussian_lr.score(x_lr_test, y_lr_test), support_lr.score(x_lr_test, y_lr_test),
knn_rf.score(x_rf_test, y_rf_test), decisiontree_rf.score(x_rf_test, y_rf_test), randomforest_rf.score(x_rf_test, y_rf_test), gaussian_rf.score(x_rf_test, y_rf_test), support_rf.score(x_rf_test, y_rf_test)]
plt.rcParams.update({'figure.figsize':(10,12), 'font.size': 36, 'font.weight' : 'bold'})
plt.plot(x,y)

In [ ]:
plt.scatter(x,y)